# ASTRA Workbench

In [1]:
import astra
import matplotlib.pyplot as plt
import numpy as np
import sys 
sys.path.append('..\\scripts\\python')
import amglib.readers as rd
import amglib.imageutils as amg
import amglib.widgets as aw

from scipy.ndimage import shift


import importlib

ModuleNotFoundError: No module named 'astra'

In [ ]:
importlib.reload(rd)
importlib.reload(aw)

## Check ASTRA

In [ ]:
astra.test()

## Load and prepare data

In [ ]:
#projselect = aw.FileSelector()
#projselect.display()

In [ ]:
#proj = rd.read_images('C:\\Users\\antoni_c\\DATA\\P20250668\\02_rawdata\\01_Config1\\01_Calibration\\cal_{0:05d}.fits',first=1, last=361)

proj = rd.read_images('C:\\Users\\antoni_c\\DATA\\P20250668\\02_rawdata\\01_Config1\\02_FlatScan\\flatscan_{0:05d}.fits',first=1, last=1125)

In [ ]:
#obselect = aw.FileSelector()
#obselect.display()

In [ ]:
#ob = rd.read_images('C:\\Users\\antoni_c\\DATA\\P20250668\\02_rawdata\\01_Config1\\01_Calibration\\ob_{0:05d}.fits',first=1, last=5).mean(axis=0)
ob = rd.read_images('C:\\Users\\antoni_c\\DATA\\P20250668\\02_rawdata\\01_Config1\\02_FlatScan\\ob_{0:05d}.fits',first=1, last=10).mean(axis=0)

In [ ]:
#dcselect = aw.FileSelector()
#dcselect.display()

In [ ]:
#dc = rd.read_images('C:\\Users\\antoni_c\\DATA\\P20250668\\02_rawdata\\01_Config1\\01_Calibration\\dc_{0:05d}.fits',first=1, last=5).mean(axis=0)
dc = rd.read_images('C:\\Users\\antoni_c\\DATA\\P20250668\\02_rawdata\\01_Config1\\02_FlatScan\\dc_{0:05d}.fits',first=1, last=10).mean(axis=0)

### Normalize

In [ ]:
ob.shape

In [ ]:
p = amg.normalizeImage(img=proj,ob=ob,dc=dc,neglog=True)

In [ ]:
orig_sino = p[:,p.shape[1]//2,:]
plt.imshow(orig_sino)
print(orig_sino.shape)


## The reconstruction

In [ ]:
#vol_geom = astra.create_vol_geom(1792, 1125, 2176)
#num_projections = 360                                                           #for calibration data
num_projections = 1125                                                           #for flatscan data

z = p.shape[0]
y = p.shape[1]
x = p.shape[2]
print(f"z = {z}")
print(f"y = {y}")
print(f"x = {x}")


#angles = np.linspace(0, 2 * np.pi, num=num_projections, endpoint=False)
angles = np.linspace(0, 2 * np.pi, num=z, endpoint=False)
import scipy.ndimage
#p_small = scipy.ndimage.zoom(p, (0.25, 0.25, 0.25), order=1) 
zoom_factor = 0.25
p_small = scipy.ndimage.zoom(p, (zoom_factor,zoom_factor,zoom_factor), order=3) 

# Distances (in voxel units)
detector_pixel_size = 0.139/zoom_factor
det_rows, det_cols = 2176, 1792
SOD = 395                  # Source to object distance
SDD = 1327                  # Source to detector distance
det_distance = SDD-SOD                  # Source to detector distance
source_distance = 395                # Source to origin distance
#vol_geom = astra.create_vol_geom(det_rows, det_cols, num_projections) 
#vol_geom = astra.create_vol_geom(y, x, z) 
vol_geom = astra.create_vol_geom(p_small.shape[1], p_small.shape[2], p_small.shape[0])

In [ ]:
plt.imshow(p_small[:,p_small.shape[1]//2,:])

In [ ]:
#detector_geom = {'DetectorRowCount': det_rows, 'DetectorColCount': det_cols, 'DetectorSpacing': (detector_pixel_size, detector_pixel_size)}                                                                                   
detector_geom = {'DetectorRowCount': p_small.shape[1], 'DetectorColCount': p_small.shape[2], 
                 'DetectorSpacing': (detector_pixel_size, detector_pixel_size)}                                                                                   

# Generate projection geometry
#proj_geom = astra.create_proj_geom('cone', detector_geom['DetectorSpacing'][0], detector_geom['DetectorSpacing'][1],
#                                   det_rows, det_cols, angles,
#                                   source_distance, det_distance)
proj_geom = astra.create_proj_geom('cone', detector_geom['DetectorSpacing'][0], detector_geom['DetectorSpacing'][1],
                                   p_small.shape[1], p_small.shape[2], angles, source_distance, det_distance)


In [ ]:
#vol_id = astra.data3d.create('-vol', vol_geom, p)
#proj_id, projections = astra.creators.create_sino3d_gpu(vol_id, proj_geom, vol_geom)

vol_id = astra.data3d.create('-vol', vol_geom, p_small)
proj_id, projections = astra.creators.create_sino3d_gpu(vol_id, proj_geom, vol_geom)


In [ ]:
## Run the reconstruction algorithm
rec_id = astra.data3d.create('-vol', vol_geom)
cfg = astra.astra_dict('FDK_CUDA')
cfg['ReconstructionDataId'] = rec_id
cfg['ProjectionDataId'] = proj_id
cfg['ProjectorId'] = astra.create_projector('cuda3d', proj_geom, vol_geom)
cfg['option'] = { 'MinConstraint': 0, 'MaxConstraint': 1 }

In [ ]:
alg_id = astra.algorithm.create(cfg)
astra.algorithm.run(alg_id)
reconstruction = astra.data3d.get(rec_id)

In [ ]:
#plt.imshow(V[100:300,100:300],interpolation="none",cmap='viridis')
#plt.imshow(reconstruction, interpolation="none", cmap='viridis')  # Εμφανίζει την ανακατασκευασμένη εικόνα V με χρωματική παλέτα 'viridis'
#plt.show()
plt.imshow(reconstruction[reconstruction.shape[0] // 2], interpolation="none", cmap='viridis')
# plt.clim(0, 500)
plt.show()
#plt.savefig("reconstruction.png", dpi=300)
plt.imsave("reconstruction_slice.png", reconstruction[reconstruction.shape[0] // 2], cmap='viridis')


In [ ]:
reconstruction.shape

In [ ]:

## Clean-up
astra.algorithm.delete(alg_id)
astra.data3d.delete([vol_id, proj_id, rec_id])
astra.projector.delete([proj_id, projections])   


In [ ]:
#plt.imshow(sinogram)

## Old staff

In [ ]:

#dx = -2.75    #     -6.561                                                      #μετατόπιση (στην κατεύθυνση των ανιχνευτών)
#sino = shift(orig_sino,shift=(0,dx), order=3)                         #μετακινεί το αρχικό σινόγραμμα orig_sino κατά dx pixels, ordeer 3ης ταξη παρεμβολη, shift μονο οριζοντια μετατοπιση
# create geometries and projector
#N = sino.shape[1]                                                     #αριθμός pixels ανά προβολή (πλάτος σινογράμματος)
#Np = sino.shape[0]                                                    #αριθμός προβολών (πλήθος γωνιών)
#arc = 2*np.pi                                                         #περιστροφη σε radian (2π για πλήρη περιστροφή)
#rot = np.deg2rad(75)                                                              #μετατόπιση γωνίας 

         #proj_geom->Δημιουργείται παράλληλη γεωμετρία προβολής   
#angles = np.linspace(0, arc, Np, endpoint=False)+rot
#d_source = 395.87
#d_detector = 1331 - d_source  # 935.13
#proj_geom = astra.create_proj_geom('parallel', 1.0, N, angles, d_source, d_detector)

#Ny = 793                                                        #πλάτος όγκου ανακατασκευής
#Nx = 890                                                        #ύψος όγκου ανακατασκευής
                                                                                                      
#vol_geom = astra.create_vol_geom(N, N)                                                                       #vol_geom->Δημιουργείται γεωμετρία όγκου, (NxN pixels)
#vol_size = int(2 * min(Ny, Nx))  # or pick a value that fits your object
#vol_geom = astra.create_vol_geom(vol_size, vol_size)
#proj_id = astra.create_projector('cuda', proj_geom, vol_geom)                                                 #proj_id->Δημιουργείται ο προτζέκτορας για CUDA


#sinogram_id = astra.data2d.create('-sino', proj_geom, data=sino)                                              #sinogram_id->Δημιουργείται το σινόγραμμα
# reconstruction volume
#recon_id = astra.data2d.create('-vol', vol_geom)        # , 0)                                                           #recon_id->Δημιουργεί έναν "άδειο" όγκο στον οποίο θα γίνει η ανακατασκευή με αρχική τιμή 0

#cfg                         = astra.astra_dict('SIRT_CUDA')           #μην πειραξεις
#cfg['ProjectorId']          = proj_id
#cfg['ProjectionDataId']     = sinogram_id
#cfg['ReconstructionDataId'] = recon_id
#cfg['option']               = { 'MinConstraint': 0, 'MaxConstraint': 1 }

#sirt_id = astra.algorithm.create(cfg)                                 

#astra.algorithm.run(sirt_id, 300)                                                                           #sirt_id->Δημιουργεί τον αλγόριθμο τον εκτελεί για 300 επαναλήψεις
#V = astra.data2d.get(recon_id)                                                                              #V->Αποθηκεύει τα δεδομένα ανακατασκευής από τον recon_id μορφή πίνακα


#plt.imshow(V[100:300,100:300],interpolation="none",cmap='viridis')
#plt.imshow(V, interpolation="none", cmap='viridis')  # Εμφανίζει την ανακατασκευασμένη εικόνα V με χρωματική παλέτα 'viridis'
#plt.show()

# garbage disposal
#astra.data2d.delete([sinogram_id, recon_id, V_exact_id])
#astra.projector.delete(proj_id)
#astra.algorithm.delete(sirt_id)






In [ ]:
#plt.imshow(sinogram)

# Playground

In [ ]:
import re
def find_first_last_indices(filenames):
    indices = []
    
    ext = filenames[0].split('.')[-1]

    pattern=r'_(0*\d+)(\.'+ext+')$'

    for filename in filenames:
        filename = filename.split('/')[-1]
        match=re.search(pattern, filename)
        if match:
            indices.append(int(match.group(1)))  # Convert to integer

    if indices:
        return min(indices), max(indices)
    else:
        return None, None  # If no matches found

In [ ]:
flist=rd.list_matching_files('D:/Kaestner/TestData/Parallel_WoodData/projections/','wood_*.tif')

In [ ]:
find_first_last_indices(flist)

In [ ]:
import os
os.path.basename('D:\\Kaestner\\TestData\\Parallel_WoodData\\projections\\dc_0002.tif')

# Center of rotation

In [ ]:
sino = p_small[:,p_small.shape[1]//2,:]
plt.imshow(sino)

In [ ]:
sino.shape

In [ ]:
plt.plot(sino[0,:])
plt.plot(sino[141,::-1])

In [ ]:
plt.plot(np.corrcoef(sino[0,:], sino[141,::-1]))

In [ ]:
# Extract the first and opposite projections
proj0 = sino[0, :]
proj180 = sino[sino.shape[0] // 2, ::-1]  # Flip the 180° projection

# Compute cross-correlation
corr = scipy.signal.correlate(proj0, proj180, mode='full')
shift = np.argmax(corr) - (len(proj0) - 1)

print(f"Estimated center shift: {shift} pixels")

# Optionally, plot the correlation
plt.figure()
plt.plot(corr)
plt.title("Cross-correlation between 0° and 180° projections")
plt.xlabel("Shift")
plt.ylabel("Correlation")
plt.show()